# Fetch Data

From https://history.geometrydash.eu/

In [10]:
import requests
import pandas as pd
import polars as pl
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

## Constants

In [3]:
BASE_URL = "https://history.geometrydash.eu"
DAILY_WEEKLY_ENDPOINT = "/api/v1/daily/"
LEVEL_RECORDS_ENDPOINT = "/api/v1/level/{level_id}/dupes/"

## ETL

### Daily and Weekly Featured + Events

In [4]:
response = requests.get(BASE_URL + DAILY_WEEKLY_ENDPOINT)
data = response.json()

In [5]:
keys = data.keys()
rows = []
for key in keys:
    dict_list = data[key]
    for d in dict_list:
        if key not in ["Weekly", "Event"]:
            d["level_type"] = "Daily"
        else:
            d["level_type"] = key
        d["level_type_details"] = key
        rows.append(d)

levels_df = pd.DataFrame(rows)
levels_df.to_csv("../data/levels.csv", index=False)

### Historical Records

In [11]:
KEEP = {
    "id", "likes", "dislikes", "downloads", "rating", "rating_sum",
    "daily_id", "feature_score", "stars", "epic",
    "cache_real_date", "record_type",
    "demon", "demon_type", "length", "objects_count",
    "real_date",
}

def fetch_level(level_id):
    url = (BASE_URL + LEVEL_RECORDS_ENDPOINT).format(level_id=level_id)
    try:
        resp = requests.get(url, timeout=30)
        data = resp.json()
    except Exception:
        return []

    records = data.get("records", [])
    slim = []
    for r in records:
        r["online_id"] = level_id
        slim.append({k: r[k] for k in KEEP if k in r})
    return slim


level_ids = levels_df["online_id"].unique().tolist()
all_records = []

with ThreadPoolExecutor(max_workers=20) as executor:
    futures = {executor.submit(fetch_level, lid): lid for lid in level_ids}
    for future in tqdm(as_completed(futures), total=len(level_ids), desc="Fetching levels"):
        all_records.extend(future.result())

Fetching levels: 100%|██████████| 3865/3865 [24:53<00:00,  2.59it/s]  


In [13]:
records_df = pl.DataFrame(all_records, infer_schema_length=None)
records_df.write_parquet("../data/records.parquet", compression_level=22)